# MapBiomas Chile Collection 2.0 — GEE exploration

Pull LULC class histograms from Google Earth Engine, then join with OCHA comuna polygons to get land-use shares per city.

**Scope:** set `FILTER_TEST_CITIES` in the config cell — `False` runs all comunas in the OCHA GPKG (~345); `True` limits to 12 HIAP-MEED Los Ríos test cities.

Prerequisites: [Earth Engine signup](https://earthengine.google.com/), `earthengine authenticate` once, `pip install earthengine-api geopandas pandas`.

In [31]:
from pathlib import Path

ASSET_ID = (
    "projects/mapbiomas-chile/assets/LULC/COLLECTION-02/CLASSIFICATIONS/"
    "classification-final/clasificacion-final-2"
)
START_YEAR = 1999
YEAR = 2023
SCALE_M = 30
OCHA_LAYER = "cl_admin_locode"
FILTER_TEST_CITIES = False  # True = 12 HIAP-MEED Los Ríos comunas only; False = all OCHA comunas (~345)


def _find_release_dir() -> Path:
    """Locate collection-02 regardless of notebook kernel cwd."""
    rel_paths = (
        Path("reviews/gee/cl-mapbiomas/releases/collection-02"),
        Path("gee/cl-mapbiomas/releases/collection-02"),
        Path("dataset-review/reviews/gee/cl-mapbiomas/releases/collection-02"),
    )
    for base in (Path.cwd(), *Path.cwd().parents):
        if base.name == "collection-02" and (base / "review.yaml").is_file():
            return base
        for rel in rel_paths:
            candidate = (base / rel).resolve()
            if (candidate / "review.yaml").is_file():
                return candidate
    raise FileNotFoundError("collection-02 release folder not found (review.yaml)")


def _find_ocha_gpkg() -> Path:
    rel = Path(
        "ocha-rolac/cl-ocha-ab/releases/2021/sample/raw_data_cl_ocha_ab.gpkg"
    )
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            base / rel,
            base / "reviews" / rel,
            base / "dataset-review" / "reviews" / rel,
        ):
            if candidate.is_file():
                return candidate.resolve()
    raise FileNotFoundError(f"OCHA GPKG not found: {rel}")


def _find_inventories_dir() -> Path:
    rel = Path("oef/hiap-meed/data/input/inventories")
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (
            base / rel,
            base / "reviews" / rel,
            base / "dataset-review" / "reviews" / rel,
        ):
            if candidate.is_dir() and any(candidate.glob("Inventory-*.csv")):
                return candidate.resolve()
    raise FileNotFoundError(f"HIAP-MEED inventories not found: {rel}")


RELEASE_DIR = _find_release_dir()
DATA_DIR = RELEASE_DIR / "data"
SAMPLE_DIR = RELEASE_DIR / "sample"
OCHA_GPKG = _find_ocha_gpkg()
INVENTORIES_DIR = _find_inventories_dir() if FILTER_TEST_CITIES else None
OUTPUT_SUFFIX = "_hiap_meed" if FILTER_TEST_CITIES else ""

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

In [32]:
import json
import ee
import geopandas as gpd
import pandas as pd

In [33]:
ee.Initialize(project="citycatalyst")

In [35]:
lulc_stack = ee.Image(ASSET_ID)

In [36]:
band_index = YEAR - START_YEAR
lulc_year = lulc_stack.select([band_index]).rename("lulc")

In [37]:
from shapely import make_valid

cities = gpd.read_file(OCHA_GPKG, layer=OCHA_LAYER)
cities = cities.to_crs(4326)

invalid = ~cities.geometry.is_valid
if invalid.any():
    cities.loc[invalid, "geometry"] = cities.loc[invalid, "geometry"].apply(make_valid)

if FILTER_TEST_CITIES:
    import unicodedata

    def _norm_name(s: str) -> str:
        s = unicodedata.normalize("NFD", str(s).lower().strip())
        s = "".join(c for c in s if unicodedata.category(c) != "Mn")
        return s.replace(" ", "").replace("_", "")

    inventory_keys = {
        _norm_name(p.stem.removeprefix("Inventory-").removesuffix("-2020"))
        for p in INVENTORIES_DIR.glob("Inventory-*.csv")
    }
    cities = cities[cities["comuna_name"].map(_norm_name).isin(inventory_keys)].copy()

In [38]:
prop_cols = ["comuna_code", "region_code", "comuna_name", "locode", "locode_name"]
cities_geojson = json.loads(cities[prop_cols + ["geometry"]].to_json())
cities_fc = ee.FeatureCollection(cities_geojson)

In [39]:
zonal = lulc_year.reduceRegions(
    collection=cities_fc,
    reducer=ee.Reducer.frequencyHistogram(),
    scale=SCALE_M,
    tileScale=4,
)

In [ ]:
raw_path = SAMPLE_DIR / f"lulc_histogram_{YEAR}{OUTPUT_SUFFIX}.geojson"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(zonal.getInfo(), f)

In [10]:
with open(raw_path, encoding="utf-8") as f:
    zonal_geojson = json.load(f)

rows = []
for feat in zonal_geojson["features"]:
    props = feat["properties"]
    hist = props.get("histogram") or {}
    for class_id, pixel_count in hist.items():
        rows.append(
            {
                "year": YEAR,
                "comuna_code": props.get("comuna_code"),
                "locode": props.get("locode"),
                "lulc_class": int(class_id),
                "pixel_count": int(pixel_count),
            }
        )

lulc_long = pd.DataFrame(rows)

In [26]:
totals = lulc_long.groupby("comuna_code", as_index=False)["pixel_count"].sum()
totals = totals.rename(columns={"pixel_count": "pixel_total"})

lulc_pct = lulc_long.merge(totals, on="comuna_code", how="left")
lulc_pct["share"] = lulc_pct["pixel_count"] / lulc_pct["pixel_total"]

In [27]:
crosswalk = pd.read_csv(DATA_DIR / "lulc_to_indicator_crosswalk.csv")
comuna_names = cities[["comuna_code", "comuna_name"]].drop_duplicates("comuna_code")

lulc_indicators = (
    lulc_pct.merge(crosswalk[["lulc_class", "indicator_group"]], on="lulc_class")
    .groupby(["year", "comuna_code", "indicator_group"], as_index=False, dropna=False)
    .agg(pixel_count=("pixel_count", "sum"), pixel_total=("pixel_total", "first"))
)
lulc_indicators["area_pct"] = (
    100 * lulc_indicators["pixel_count"] / lulc_indicators["pixel_total"]
).round(1)
lulc_indicators = lulc_indicators.merge(comuna_names, on="comuna_code", how="left")

out_cols = ["comuna_code", "comuna_name", "year", "indicator_group", "area_pct"]
lulc_indicators[out_cols].to_csv(
    DATA_DIR / "city_landuse_indicators.csv",
    index=False,
)
lulc_indicators[out_cols].to_csv(
    SAMPLE_DIR / f"lulc_indicators_by_city_{YEAR}{OUTPUT_SUFFIX}.csv",
    index=False,
)

In [41]:
# Optional: MapBiomas class-level detail (with labels)
legend = pd.read_csv(DATA_DIR / "legend_collection_02.csv")
legend["lulc_label"] = (
    legend["class_level_3_en"].replace("", pd.NA)
    .fillna(legend["class_level_2_en"].replace("", pd.NA))
    .fillna(legend["class_level_1_en"])
)
lulc_pct.merge(legend[["lulc_class", "lulc_label"]], on="lulc_class").merge(
    comuna_names, on="comuna_code"
).to_csv(SAMPLE_DIR / f"lulc_share_by_city_{YEAR}{OUTPUT_SUFFIX}.csv", index=False)

In [42]:
threshold_df = pd.read_csv(DATA_DIR / "indicator_thresholds.csv")


def bucket_for(threshold_df, indicator, value):
    sub = threshold_df[threshold_df.indicator == indicator].sort_values("lower_pct")
    for _, row in sub.iterrows():
        if row.bucket == "very_high" and value >= row.lower_pct:
            return row.bucket
        if row.lower_pct <= value < row.upper_pct:
            return row.bucket
    return None


def bucket_label(bucket):
    return None if bucket is None else bucket.replace("_", " ")


city_geo = (
    cities[["comuna_code", "region_code", "region_name", "comuna_name"]]
    .drop_duplicates("comuna_code")
    .assign(
        region=lambda d: d["region_code"].str.removeprefix("CL").astype(int),
        comuna=lambda d: d["comuna_code"].str.removeprefix("CL").astype(int),
        region_nombre=lambda d: d["region_name"].str.replace(
            r"^Región de ", "", regex=True
        ),
        comuna_nombre=lambda d: d["comuna_name"],
    )
)

raw_rows = []
for _, row in lulc_indicators.iterrows():
    attribute_type = f"{row['indicator_group']}_share"
    if attribute_type not in threshold_df["indicator"].values:
        continue
    bucket = bucket_for(threshold_df, attribute_type, row["area_pct"])
    if bucket is None:
        continue
    geo = city_geo.loc[city_geo["comuna_code"] == row["comuna_code"]].iloc[0]
    raw_rows.append(
        {
            "region": geo["region"],
            "comuna": geo["comuna"],
            "region_nombre": geo["region_nombre"],
            "comuna_nombre": geo["comuna_nombre"],
            "attribute_type": attribute_type,
            "attribute_value": row["area_pct"],
            "attribute_units": "percent",
            "attribute_category": bucket_label(bucket),
        }
    )

raw_data_cl_mapbiomas_lulc = pd.DataFrame(raw_rows)
raw_data_cl_mapbiomas_lulc.to_csv(
    DATA_DIR / f"raw_data_cl_mapbiomas_lulc{OUTPUT_SUFFIX}.csv",
    index=False,
)